In [2]:
import random
import time
import copy

class SchoolOptimizer:
    def __init__(self, pop_size=100, mutation_rate=0.2, elite_size=10, generations=200):
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.elite_size = elite_size
        self.generations = generations
        
        # --- DONNÉES DU PROBLÈME ---
        
        # Liste des matières (M1 à M10) mappées de 0 à 9
        self.subjects = [
            "M1 (Maths)", "M2 (Physique)", "M3 (Chimie)", "M4 (Bio)", "M5 (Info)",
            "M6 (Hist-Géo)", "M7 (Langues)", "M8 (Arts)", "M9 (Musique)", "M10 (EPS)"
        ]
        
        # Matrice des Fréquences (F) - Copiée depuis l'énoncé
        # F[i][j] = fréquence de déplacement de la matière i vers la matière j
        self.flow_matrix = [
            [0.00, 0.15, 0.10, 0.05, 0.20, 0.08, 0.07, 0.03, 0.02, 0.10],
            [0.12, 0.00, 0.18, 0.12, 0.05, 0.04, 0.06, 0.02, 0.01, 0.10],
            [0.08, 0.16, 0.00, 0.14, 0.06, 0.05, 0.04, 0.03, 0.02, 0.12],
            [0.04, 0.10, 0.12, 0.00, 0.08, 0.10, 0.06, 0.08, 0.04, 0.08],
            [0.18, 0.04, 0.05, 0.06, 0.00, 0.12, 0.10, 0.05, 0.03, 0.07],
            [0.06, 0.03, 0.04, 0.08, 0.10, 0.00, 0.15, 0.10, 0.08, 0.06],
            [0.05, 0.05, 0.03, 0.05, 0.08, 0.14, 0.00, 0.12, 0.10, 0.08],
            [0.02, 0.02, 0.02, 0.06, 0.04, 0.08, 0.10, 0.00, 0.15, 0.11],
            [0.01, 0.01, 0.01, 0.03, 0.02, 0.06, 0.08, 0.12, 0.00, 0.16],
            [0.09, 0.08, 0.10, 0.06, 0.06, 0.05, 0.06, 0.09, 0.13, 0.00]
        ]

        # Matrice des Distances (D) - Copiée depuis l'énoncé
        # D[i][j] = distance entre la Position i et la Position j (Indices 0-9 pour P1-P10)
        self.dist_matrix = [
            [0, 20, 40, 60, 100, 85, 95, 110, 50, 65],
            [20, 0, 20, 40, 85, 70, 80, 95, 40, 45],
            [40, 20, 0, 20, 95, 80, 70, 85, 55, 25],
            [60, 40, 20, 0, 110, 95, 85, 70, 75, 15],
            [100, 85, 95, 110, 0, 20, 40, 60, 65, 95],
            [85, 70, 80, 95, 20, 0, 20, 40, 45, 80],
            [95, 80, 70, 85, 40, 20, 0, 20, 55, 70],
            [110, 95, 85, 70, 60, 40, 20, 0, 75, 55],
            [50, 40, 55, 75, 65, 45, 55, 75, 0, 70],
            [65, 45, 25, 15, 95, 80, 70, 55, 70, 0]
        ]

    # --- 1. REPRÉSENTATION ---
    def create_individual(self):
        """Crée une permutation aléatoire des 10 matières [0, 1, ... 9]"""
        individual = list(range(10))
        random.shuffle(individual)
        return individual

    # --- 2. ÉVALUATION (FITNESS) ---
    def calculate_cost(self, solution):
        """Calcule le coût total (Distance pondérée + Pénalités)"""
        
        # A. Distance totale pondérée
        total_distance = 0
        for p1 in range(10): # Pour chaque position P1
            for p2 in range(10): # Pour chaque position P2
                if p1 == p2: continue
                
                subject_i = solution[p1] # Quelle matière est en P1 ?
                subject_j = solution[p2] # Quelle matière est en P2 ?
                
                # Flux * Distance
                flow = self.flow_matrix[subject_i][subject_j]
                dist = self.dist_matrix[p1][p2]
                total_distance += flow * dist
        
        # B. Pénalités (Contraintes)
        # Rappel index: M1=0, M2=1 ... M10=9
        # Rappel index positions: P1=0 ... P10=9
        penalties = 0
        
        # P1: EPS (M10/Idx 9) doit être en P8(7) ou P10(9)
        pos_m10 = solution.index(9)
        if pos_m10 != 7 and pos_m10 != 9:
            penalties += 1000
            
        # P2: Musique (M9/Idx 8) doit être en P9(8)
        pos_m9 = solution.index(8)
        if pos_m9 != 8:
            penalties += 500
            
        # P3: Arts (M8/Idx 7) doit être en P8(7) ou P10(9)
        pos_m8 = solution.index(7)
        if pos_m8 != 7 and pos_m8 != 9:
            penalties += 300
            
        # P4: Chimie (M3/Idx 2) doit être en P3(2) ou P7(6)
        pos_m3 = solution.index(2)
        if pos_m3 != 2 and pos_m3 != 6:
            penalties += 200
            
        # P5: Info (M5/Idx 4) doit être en P3(2) ou P7(6)
        pos_m5 = solution.index(4)
        if pos_m5 != 2 and pos_m5 != 6:
            penalties += 200

        return total_distance, penalties

    def get_fitness(self, solution):
        dist, pen = self.calculate_cost(solution)
        # On veut minimiser le coût, donc maximiser 1/Coût
        # On ajoute 1 pour éviter la division par zéro
        return 10000 / (1 + dist + pen)

    # --- 3. SÉLECTION ---
    def tournament_selection(self, population, fitnesses, k=3):
        """Sélection par tournoi"""
        selected = random.sample(list(zip(population, fitnesses)), k)
        # Retourne celui avec le meilleur fitness
        return max(selected, key=lambda item: item[1])[0]

    # --- 4. CROISEMENT (ORDER CROSSOVER - OX1) ---
    def crossover(self, parent1, parent2):
        """Croisement spécifique aux permutations pour éviter les doublons"""
        size = len(parent1)
        start, end = sorted(random.sample(range(size), 2))
        
        child = [-1] * size
        
        # 1. Copier le segment du parent 1
        child[start:end] = parent1[start:end]
        
        # 2. Remplir le reste avec les gènes du parent 2 (dans l'ordre)
        current_p2_idx = 0
        for i in range(size):
            if child[i] == -1:
                # Trouver le prochain gène de P2 qui n'est pas déjà dans Child
                while parent2[current_p2_idx] in child:
                    current_p2_idx += 1
                child[i] = parent2[current_p2_idx]
                
        return child

    # --- 5. MUTATION (SWAP) ---
    def mutate(self, individual):
        """Échange deux positions aléatoirement"""
        if random.random() < self.mutation_rate:
            idx1, idx2 = random.sample(range(len(individual)), 2)
            individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
        return individual

    # --- MAIN LOOP ---
    def run(self):
        # 1. Initialisation
        population = [self.create_individual() for _ in range(self.pop_size)]
        best_overall = None
        best_score_overall = -1
        
        print(f"Début de l'optimisation sur {self.generations} générations...")
        print("-" * 50)

        start_time = time.time()

        for gen in range(self.generations):
            # 2. Évaluation
            fitnesses = [self.get_fitness(ind) for ind in population]
            
            # Tracking du meilleur
            max_fit = max(fitnesses)
            idx_best = fitnesses.index(max_fit)
            current_best = population[idx_best]
            
            if max_fit > best_score_overall:
                best_score_overall = max_fit
                best_overall = copy.deepcopy(current_best)
                dist, pen = self.calculate_cost(best_overall)
                print(f"Gen {gen:3d} : Nouveau record ! Coût Total = {dist + pen:.1f} (Dist: {dist:.1f}, Pen: {pen})")

            # 3. Nouvelle population
            new_pop = []
            
            # A. Élitisme (On garde les meilleurs)
            # On trie la population par fitness décroissant
            sorted_pop = [x for _, x in sorted(zip(fitnesses, population), key=lambda pair: pair[0], reverse=True)]
            new_pop.extend(sorted_pop[:self.elite_size])
            
            # B. Reproduction
            while len(new_pop) < self.pop_size:
                p1 = self.tournament_selection(population, fitnesses)
                p2 = self.tournament_selection(population, fitnesses)
                child = self.crossover(p1, p2)
                child = self.mutate(child)
                new_pop.append(child)
            
            population = new_pop

        end_time = time.time()
        
        # --- RÉSULTATS FINAUX ---
        dist, pen = self.calculate_cost(best_overall)
        print("-" * 50)
        print("OPTIMISATION TERMINÉE")
        print(f"Temps de calcul : {end_time - start_time:.2f} secondes")
        print(f"Meilleur arrangement trouvé (Coût: {dist+pen:.1f})")
        print(f"  > Distance pondérée : {dist:.1f}")
        print(f"  > Pénalités : {pen} (Objectif = 0)")
        print("-" * 50)
        
        self.display_solution(best_overall)

    def display_solution(self, solution):
        print("PLAN DES SALLES :")
        for i in range(10):
            subject_idx = solution[i]
            subject_name = self.subjects[subject_idx]
            pos_name = f"P{i+1}"
            
            # Vérification visuelle des contraintes
            status = "OK"
            # Exemple de check rapide pour l'affichage
            if subject_idx == 9 and (i != 7 and i != 9): status = "!!! NON RESPECTÉ (EPS)"
            if subject_idx == 8 and i != 8: status = "!!! NON RESPECTÉ (Musique)"
            
            print(f"{pos_name} : {subject_name} \t-> {status}")

# --- Lancement ---
if __name__ == "__main__":
    # Paramètres ajustables pour l'expérience
    optimizer = SchoolOptimizer(
        pop_size=200,       # Taille de la population
        mutation_rate=0.3,  # Taux de mutation (assez élevé pour éviter les optimums locaux)
        elite_size=20,      # Elitisme
        generations=150     # Nombre de générations
    )
    optimizer.run()

Début de l'optimisation sur 150 générations...
--------------------------------------------------
Gen   0 : Nouveau record ! Coût Total = 650.3 (Dist: 450.3, Pen: 200)
Gen   1 : Nouveau record ! Coût Total = 624.0 (Dist: 424.0, Pen: 200)
Gen   2 : Nouveau record ! Coût Total = 612.5 (Dist: 412.5, Pen: 200)
Gen   3 : Nouveau record ! Coût Total = 427.9 (Dist: 427.9, Pen: 0)
Gen   4 : Nouveau record ! Coût Total = 400.8 (Dist: 400.8, Pen: 0)
Gen   5 : Nouveau record ! Coût Total = 397.6 (Dist: 397.6, Pen: 0)
Gen   7 : Nouveau record ! Coût Total = 397.3 (Dist: 397.3, Pen: 0)
Gen   8 : Nouveau record ! Coût Total = 392.6 (Dist: 392.6, Pen: 0)
Gen   9 : Nouveau record ! Coût Total = 387.3 (Dist: 387.3, Pen: 0)
Gen  10 : Nouveau record ! Coût Total = 386.8 (Dist: 386.8, Pen: 0)
--------------------------------------------------
OPTIMISATION TERMINÉE
Temps de calcul : 1.46 secondes
Meilleur arrangement trouvé (Coût: 386.8)
  > Distance pondérée : 386.8
  > Pénalités : 0 (Objectif = 0)
------